# Titanic Rescue Mission - Kaggle Solution

This notebook builds a complete, reproducible solution for Kaggle's Titanic competition. It loads the training and test files, explores the data, engineers Titanic-specific features, compares several scikit-learn models with cross-validation, trains the best model, and writes `submission.csv`.

Run all cells after attaching the Titanic dataset in Kaggle, or place `train.csv` and `test.csv` in this folder / `dataset_raw` when running locally.

In [ ]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='Set2')

## 1. Load the data

The helper below works in Kaggle and in a local project folder. Kaggle usually stores the competition files under `/kaggle/input/titanic/`.

In [ ]:
def find_competition_file(filename):
    search_roots = [Path('/kaggle/input'), Path.cwd(), Path.cwd() / 'dataset_raw']
    for root in search_roots:
        if not root.exists():
            continue
        direct_match = root / filename
        if direct_match.exists():
            return direct_match
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    raise FileNotFoundError(f'Could not find {filename}. Attach the Titanic dataset or place it in the project folder.')

train_path = find_competition_file('train.csv')
test_path = find_competition_file('test.csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print('Train path:', train_path)
print('Test path:', test_path)
print('Train shape:', train.shape)
print('Test shape:', test.shape)
display(train.head())
display(test.head())

## 2. Quick data checks

In [ ]:
display(train.info())
display(train.describe(include='all').T)

missing = pd.DataFrame({
    'train_missing': train.isna().sum(),
    'test_missing': test.isna().sum()
}).fillna(0).astype(int)
display(missing[missing.sum(axis=1) > 0].sort_values('train_missing', ascending=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
sns.countplot(data=train, x='Survived', ax=axes[0, 0])
sns.barplot(data=train, x='Sex', y='Survived', ax=axes[0, 1])
sns.barplot(data=train, x='Pclass', y='Survived', ax=axes[0, 2])
sns.histplot(data=train, x='Age', hue='Survived', kde=True, bins=30, ax=axes[1, 0])
sns.barplot(data=train, x='Embarked', y='Survived', ax=axes[1, 1])
sns.histplot(data=train, x='Fare', hue='Survived', bins=30, ax=axes[1, 2])

for ax in axes.ravel():
    ax.set_title(ax.get_xlabel() or 'Survival')
plt.tight_layout()
plt.show()

## 3. Feature engineering

The custom transformer creates reusable features from names, cabins, tickets, family counts, and fares. It avoids using the target column, so the same transformation works for train and test data.

In [ ]:
class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, smoothing=8):
        self.smoothing = smoothing

    def _surname(self, name_series):
        return name_series.fillna('Unknown').astype(str).str.split(',', n=1).str[0].str.strip()

    def _ticket_key(self, ticket_series):
        return ticket_series.fillna('Unknown').astype(str).str.replace(r'\s+', '', regex=True)

    def _make_rate_map(self, keys, y):
        frame = pd.DataFrame({'key': keys, 'target': y})
        grouped = frame.groupby('key')['target'].agg(['mean', 'count'])
        smoothed = (grouped['mean'] * grouped['count'] + self.global_survival_rate_ * self.smoothing) / (grouped['count'] + self.smoothing)
        return smoothed.to_dict(), grouped['count'].to_dict()

    def fit(self, X, y=None):
        if y is None:
            self.global_survival_rate_ = 0.5
            self.surname_rate_map_, self.surname_count_map_ = {}, {}
            self.ticket_rate_map_, self.ticket_count_map_ = {}, {}
            return self

        df = X.copy()
        target = pd.Series(y, index=df.index).astype(float)
        self.global_survival_rate_ = float(target.mean())
        self.surname_rate_map_, self.surname_count_map_ = self._make_rate_map(self._surname(df['Name']), target)
        self.ticket_rate_map_, self.ticket_count_map_ = self._make_rate_map(self._ticket_key(df['Ticket']), target)
        return self

    def _leave_one_out_rate(self, keys, target):
        frame = pd.DataFrame({'key': keys, 'target': target})
        group_sum = frame.groupby('key')['target'].transform('sum')
        group_count = frame.groupby('key')['target'].transform('count')
        loo_count = group_count - 1
        loo_sum = group_sum - target
        rate = (loo_sum + self.global_survival_rate_ * self.smoothing) / (loo_count + self.smoothing)
        return rate.where(loo_count > 0, self.global_survival_rate_)

    def fit_transform(self, X, y=None, **fit_params):
        self.fit(X, y)
        df = self.transform(X)
        if y is None:
            return df

        original = X.copy()
        target = pd.Series(y, index=original.index).astype(float)
        surname_rate = self._leave_one_out_rate(self._surname(original['Name']), target)
        ticket_rate = self._leave_one_out_rate(self._ticket_key(original['Ticket']), target)
        df['SurnameSurvivalRate'] = surname_rate.to_numpy()
        df['TicketSurvivalRate'] = ticket_rate.to_numpy()
        df['GroupSurvivalRate'] = df[['SurnameSurvivalRate', 'TicketSurvivalRate']].mean(axis=1)
        return df

    def transform(self, X):
        df = X.copy()
        surname = self._surname(df['Name'])
        ticket_key = self._ticket_key(df['Ticket'])

        df['SurnameSurvivalRate'] = surname.map(self.surname_rate_map_).fillna(self.global_survival_rate_)
        df['SurnameCount'] = surname.map(self.surname_count_map_).fillna(0)
        df['TicketSurvivalRate'] = ticket_key.map(self.ticket_rate_map_).fillna(self.global_survival_rate_)
        df['TicketGroupSize'] = ticket_key.map(self.ticket_count_map_).fillna(0)
        df['GroupSurvivalRate'] = df[['SurnameSurvivalRate', 'TicketSurvivalRate']].mean(axis=1)
        df['NameLength'] = df['Name'].fillna('').astype(str).str.len()

        df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
        title_map = {
            'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
            'Lady': 'Rare', 'Countess': 'Rare', 'the Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare', 'Don': 'Rare',
            'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare', 'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
        }
        df['Title'] = df['Title'].replace(title_map).fillna('Unknown')

        df['FamilySize'] = df['SibSp'].fillna(0) + df['Parch'].fillna(0) + 1
        df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
        df['SmallFamily'] = df['FamilySize'].between(2, 4).astype(int)
        df['LargeFamily'] = (df['FamilySize'] >= 5).astype(int)

        df['Deck'] = df['Cabin'].fillna('Unknown').astype(str).str[0]
        df.loc[df['Deck'] == 'U', 'Deck'] = 'Unknown'
        df['HasCabin'] = df['Cabin'].notna().astype(int)

        ticket = df['Ticket'].fillna('Unknown').astype(str)
        df['TicketPrefix'] = ticket.str.replace(r'[\d\.\/\s]+', '', regex=True).replace('', 'None')
        df['TicketLength'] = ticket.str.len()

        df['FarePerPerson'] = df['Fare'] / df['FamilySize'].replace(0, 1)
        df['AgeTimesClass'] = df['Age'] * df['Pclass']
        df['FareLog'] = np.log1p(df['Fare'])

        df = df.drop(columns=['Name', 'Ticket', 'Cabin'], errors='ignore')
        return df

In [ ]:
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

numeric_features = [
    'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'SmallFamily',
    'LargeFamily', 'HasCabin', 'TicketLength', 'FarePerPerson', 'AgeTimesClass', 'FareLog',
    'SurnameSurvivalRate', 'SurnameCount', 'TicketSurvivalRate', 'TicketGroupSize',
    'GroupSurvivalRate', 'NameLength'
]
categorical_features = ['Sex', 'Embarked', 'Title', 'Deck', 'TicketPrefix']

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', encoder)
])

preprocessor = Pipeline(steps=[
    ('features', TitanicFeatureEngineer()),
    ('columns', ColumnTransformer(transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]))
])

X = train.drop(columns=['Survived'])
y = train['Survived'].astype(int)

## 4. Compare models with cross-validation

Kaggle scores Titanic submissions by accuracy, so the validation metric here is accuracy.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=3000, C=0.8, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=600, max_depth=6, min_samples_leaf=2, max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=700, max_depth=7, min_samples_leaf=2, max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=250, learning_rate=0.035, max_depth=3, min_samples_leaf=3, random_state=RANDOM_STATE
    ),
    'Support Vector Machine': SVC(C=1.4, gamma='scale', probability=True, random_state=RANDOM_STATE
    )
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    results.append({'model': name, 'mean_accuracy': scores.mean(), 'std_accuracy': scores.std()})
    print(f'{name:20s} | accuracy: {scores.mean():.4f} +/- {scores.std():.4f}')

results_df = pd.DataFrame(results).sort_values('mean_accuracy', ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='mean_accuracy', y='model', orient='h')
plt.xlim(max(0.70, results_df['mean_accuracy'].min() - 0.03), min(1.0, results_df['mean_accuracy'].max() + 0.03))
plt.xlabel('Mean CV Accuracy')
plt.ylabel('Model')
plt.title('Titanic Model Comparison')
plt.tight_layout()
plt.show()

## 5. Validate the selected model

A soft-voting ensemble of the strongest model families usually performs well on this dataset without heavy tuning.

In [ ]:
ensemble_model = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=3000, C=0.8, random_state=RANDOM_STATE)),
        ('rf', RandomForestClassifier(
            n_estimators=600, max_depth=6, min_samples_leaf=2, max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1
        )),
        ('gb', GradientBoostingClassifier(
            n_estimators=250, learning_rate=0.035, max_depth=3, min_samples_leaf=3, random_state=RANDOM_STATE
        )),
        ('svc', SVC(C=1.4, gamma='scale', probability=True, random_state=RANDOM_STATE))
    ],
    voting='soft',
    weights=[1, 2, 3, 1]
)

final_pipeline = Pipeline(steps=[('preprocess', preprocessor), ('model', ensemble_model)])

ensemble_scores = cross_val_score(final_pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print(f'Ensemble CV accuracy: {ensemble_scores.mean():.4f} +/- {ensemble_scores.std():.4f}')

oof_probabilities = cross_val_predict(final_pipeline, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
threshold_scores = []
for threshold in np.linspace(0.35, 0.65, 61):
    threshold_predictions = (oof_probabilities >= threshold).astype(int)
    threshold_scores.append((threshold, accuracy_score(y, threshold_predictions)))

best_threshold, best_threshold_accuracy = max(threshold_scores, key=lambda item: item[1])
oof_predictions = (oof_probabilities >= best_threshold).astype(int)

print(f'Best OOF threshold: {best_threshold:.3f}')
print(f'Out-of-fold accuracy at best threshold: {best_threshold_accuracy:.4f}')
print('\nClassification report:')
print(classification_report(y, oof_predictions))

cm = confusion_matrix(y, oof_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Out-of-Fold Confusion Matrix')
plt.show()

## 6. Train on all data and create Kaggle submission

In [ ]:
final_pipeline.fit(X, y)
test_probabilities = final_pipeline.predict_proba(test)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_predictions
})

submission_path = Path('submission.csv')
submission.to_csv(submission_path, index=False)

print('Saved:', submission_path.resolve())
print('Submission shape:', submission.shape)
display(submission.head(10))
display(submission['Survived'].value_counts(normalize=True).rename('prediction_share'))

## 7. Optional: inspect feature names

This is useful for explaining the model after the notebook runs.

In [ ]:
fitted_preprocessor = final_pipeline.named_steps['preprocess']
column_transformer = fitted_preprocessor.named_steps['columns']

try:
    feature_names = column_transformer.get_feature_names_out()
    print('Number of model features:', len(feature_names))
    print(feature_names[:40])
except Exception as exc:
    print('Feature names are not available in this sklearn version:', exc)